# ST3 — Energy Policy: Data Ingestion & Cleaning
**Team 7 Lambda | Phase 1 | Feb 27 – Mar 2**

**Goal:** Pull, clean, and save two data sources:
1. OWID Energy Data — energy mix by country/year (auto-downloadable)
2. RFF World Carbon Pricing Database — carbon tax / ETS prices by country/year

**Causal role:** ST3 quantifies asymmetric national energy policies that amplify downstream mineral demand distortions (ST2 → ST1 → ST3 → ST4).

**Output:** `data/processed/st3_energy.parquet`, `data/processed/st3_carbon_price.parquet`

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('../src')   # src/config.py and src/utils.py

import pandas as pd
import numpy as np
import requests
from pathlib import Path

from config import FOCAL_COUNTRIES, START_YEAR, END_YEAR, ST3_RAW, ST3_PROC
from utils import normalize_countries, save_parquet, set_style, log

set_style()
log.info('ST3 setup complete.')

17:02:22 [INFO] ST3 setup complete.


## 3A — OWID Energy Data

**Source:** https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv  
**Auto-download:** Yes — the cell below fetches it directly from GitHub.  
**Key columns:** `renewables_share_energy`, `fossil_share_energy`, `nuclear_share_energy`, `solar_electricity`, `wind_electricity`

Coverage: ~200 countries, 1965–2023, annual.

In [2]:
# ── Download OWID Energy Data ───────────────────────────────────────────────
OWID_URL  = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
OWID_PATH = ST3_RAW / "owid-energy-data.csv"   # data/raw/ST3/

if not OWID_PATH.exists():
    log.info("Downloading OWID energy data (~10 MB)...")
    try:
        r = requests.get(OWID_URL, timeout=120)
        r.raise_for_status()
        OWID_PATH.write_bytes(r.content)
        log.info(f"Saved OWID CSV: {OWID_PATH}")
    except Exception as e:
        log.error(f"OWID download failed: {e}")
else:
    log.info(f"OWID file exists: {OWID_PATH}")

17:02:22 [INFO] Downloading OWID energy data (~10 MB)...
17:02:23 [INFO] Saved OWID CSV: /Users/taruntheegela/Desktop/lambda/data/raw/ST3/owid-energy-data.csv


In [3]:
# ── Load and Clean OWID Energy Data ────────────────────────────────────────

# Columns we need from the OWID dataset
OWID_COLS = [
    "country", "year",
    "renewables_share_energy",     # % of primary energy from renewables
    "fossil_share_energy",         # % from fossil fuels
    "nuclear_share_energy",        # % from nuclear
    "solar_electricity",           # TWh solar power generation
    "wind_electricity",            # TWh wind power generation
    "hydro_electricity",           # TWh hydro power generation
    "electricity_generation",      # TWh total electricity generated
    "primary_energy_consumption",  # TWh total primary energy
]


def load_owid(path: Path) -> pd.DataFrame:
    """
    Load OWID energy CSV; filter to analysis window and non-aggregate entities.
    Normalizes country names to ISO-3. Drops continental/world aggregates.

    Args:
        path: Path to owid-energy-data.csv

    Returns:
        DataFrame with OWID_COLS plus 'iso3'. Empty DataFrame if file missing.

    Usage:
        owid_df = load_owid(OWID_PATH)
    """
    if not path.exists():
        log.warning(f"OWID file not found at {path}. Run the download cell first.")
        return pd.DataFrame()

    # Only load columns that exist in the file to avoid KeyErrors
    available = pd.read_csv(path, nrows=0).columns.tolist()
    cols_to_load = [c for c in OWID_COLS if c in available]
    missing = set(OWID_COLS) - set(cols_to_load)
    if missing:
        log.warning(f"OWID columns not present in file: {missing}")

    raw = pd.read_csv(path, usecols=cols_to_load, low_memory=False)

    # Filter analysis window
    raw["year"] = pd.to_numeric(raw["year"], errors="coerce")
    raw = raw[(raw["year"] >= START_YEAR) & (raw["year"] <= END_YEAR)].copy()

    # Normalize country → ISO-3 (drops world/continent aggregates that can't map)
    raw = normalize_countries(raw, "country")
    raw = raw.dropna(subset=["iso3"])

    # Coerce numeric columns
    numeric_cols = [c for c in cols_to_load if c not in ("country", "year")]
    for col in numeric_cols:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    log.info(f"OWID loaded: {raw.shape} | unique countries: {raw['iso3'].nunique()}")
    return raw.reset_index(drop=True)


owid_df = load_owid(OWID_PATH)

if not owid_df.empty:
    display(owid_df[["iso3", "country", "year",
                     "renewables_share_energy", "fossil_share_energy",
                     "solar_electricity", "wind_electricity"]].head(10))

17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'ASEAN (Ember)'
17:02:24 [WARNING] Could not map country: 'Africa (EI)'
17:02:24 [WARNING] Could not map country: 'Africa (EI)'
17:02:24 [WARNING] Could not map country: 'Africa (EI)'
17:02:24 [WARNING] Could not map country: 'Africa (EI)'
17:02:24 [WARNING] Cou

,iso3,country,year,renewables_share_energy,fossil_share_energy,solar_electricity,wind_electricity
0,AFG,Afghanistan,2010,NaN,NaN,0.00,0.0
1,AFG,Afghanistan,2011,NaN,NaN,0.00,0.0
2,AFG,Afghanistan,2012,NaN,NaN,0.03,0.0
3,AFG,Afghanistan,2013,NaN,NaN,0.03,0.0
4,AFG,Afghanistan,2014,NaN,NaN,0.03,0.0
5,AFG,Afghanistan,2015,NaN,NaN,0.03,0.0
6,AFG,Afghanistan,2016,NaN,NaN,0.04,0.0
7,AFG,Afghanistan,2017,NaN,NaN,0.04,0.0
8,AFG,Afghanistan,2018,NaN,NaN,0.04,0.0
9,AFG,Afghanistan,2019,NaN,NaN,0.06,0.0


In [4]:
# ── Build st3_energy Schema ─────────────────────────────────────────────────

def build_st3_energy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshape OWID data into the st3_energy target schema.
    Renames columns, forward-fills gaps up to 2 periods (per project rules),
    and returns cleaned DataFrame.

    Target schema:
        iso3, year, renewables_share, fossil_share, nuclear_share,
        solar_twh, wind_twh, hydro_twh, electricity_twh, primary_energy_twh

    Args:
        df: output of load_owid()

    Returns:
        Cleaned DataFrame in target schema. Empty stub if input is empty.

    Usage:
        st3_energy = build_st3_energy(owid_df)
    """
    TARGET_COLS = ["iso3", "year", "renewables_share", "fossil_share",
                   "nuclear_share", "solar_twh", "wind_twh",
                   "hydro_twh", "electricity_twh", "primary_energy_twh"]

    if df.empty:
        return pd.DataFrame(columns=TARGET_COLS)

    rename_map = {
        "renewables_share_energy":    "renewables_share",
        "fossil_share_energy":        "fossil_share",
        "nuclear_share_energy":       "nuclear_share",
        "solar_electricity":          "solar_twh",
        "wind_electricity":           "wind_twh",
        "hydro_electricity":          "hydro_twh",
        "electricity_generation":     "electricity_twh",
        "primary_energy_consumption": "primary_energy_twh",
    }

    # Only rename columns that actually exist
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    out = df[["iso3", "year"] + list(existing.keys())].rename(columns=existing)

    # Forward-fill gaps — max 2 periods (project hard rule)
    out = out.sort_values(["iso3", "year"])
    numeric_cols = [c for c in out.columns if c not in ("iso3", "year")]
    out[numeric_cols] = (
        out.groupby("iso3")[numeric_cols]
        .transform(lambda s: s.ffill(limit=2))
    )

    # Ensure all target cols present (fill missing with NaN)
    for col in TARGET_COLS:
        if col not in out.columns:
            out[col] = np.nan

    log.info(f"st3_energy built: {out.shape}")
    return out[TARGET_COLS].reset_index(drop=True)


st3_energy = build_st3_energy(owid_df)

if not st3_energy.empty:
    log.info(f"Countries with >50% renewables share (latest year): "
             f"{st3_energy[st3_energy['renewables_share'] > 50]['iso3'].nunique()}")
    display(st3_energy.describe().round(2))

17:02:42 [INFO] st3_energy built: (2793, 10)
17:02:42 [INFO] Countries with >50% renewables share (latest year): 3


,year,renewables_share,fossil_share,nuclear_share,solar_twh,wind_twh,hydro_twh,electricity_twh,primary_energy_twh
count,2793.00,1039.00,1039.00,1039.00,2739.00,2697.00,2692.00,2740.00,2779.00
mean,2016.00,12.84,83.29,3.87,2.24,5.15,19.43,121.20,744.13
std,3.74,14.80,16.96,7.39,15.65,32.53,89.16,555.85,3234.25
min,2010.00,0.00,13.87,0.00,0.00,0.00,0.00,0.00,0.00
25%,2013.00,2.90,75.41,0.00,0.00,0.00,0.00,0.91,6.44
50%,2016.00,7.70,87.99,0.00,0.01,0.00,0.75,9.16,46.00
75%,2019.00,18.10,96.08,4.58,0.16,0.39,7.82,52.73,327.90
max,2022.00,86.13,100.00,40.25,427.27,762.67,1321.71,8848.71,44516.49


## 3B — RFF World Carbon Pricing Database

**Source:** https://github.com/g-dolphin/WorldCarbonPricingDatabase  
**Auto-download:** Yes — fetched directly from GitHub raw.  
**Key columns:** `jurisdiction` (country), `year`, `tax` (carbon tax rate), ETS price  

> **Note on currency:** Carbon tax rates are reported in domestic currency per tCO₂e.  
> This notebook captures relative magnitudes and binary presence (`has_carbon_price`).  
> USD conversion requires historical exchange rates (out of scope for ingestion phase).

Coverage: ~45 jurisdictions, 1990–2022.

In [ ]:
# ── Download RFF Carbon Pricing Database ───────────────────────────────────
# Repo restructured: now individual per-country CSV files under
# _dataset/data/CO2/national/wcpd_co2_{Country}.csv
# We fetch the file list via GitHub API, then download each country.

RFF_DIR      = ST3_RAW / "rff"
RFF_COMBINED = ST3_RAW / "WorldCarbonPricingDatabase.csv"
RFF_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_API = "https://api.github.com/repos/g-dolphin/WorldCarbonPricingDatabase/contents/_dataset/data/CO2/national"
RAW_BASE   = "https://raw.githubusercontent.com/g-dolphin/WorldCarbonPricingDatabase/main/_dataset/data/CO2/national"

if not RFF_COMBINED.exists():
    log.info("Fetching RFF country file list from GitHub API...")
    try:
        resp = requests.get(GITHUB_API, timeout=30)
        resp.raise_for_status()
        files = [f["name"] for f in resp.json() if f["name"].endswith(".csv")]
        log.info(f"Found {len(files)} country files. Downloading...")

        parts = []
        for i, fname in enumerate(files):
            try:
                r = requests.get(f"{RAW_BASE}/{fname}", timeout=30)
                if r.status_code == 200:
                    import io
                    parts.append(pd.read_csv(io.StringIO(r.text)))
                if (i + 1) % 20 == 0:
                    log.info(f"  Downloaded {i+1}/{len(files)} files...")
            except Exception as e:
                log.warning(f"  Skipped {fname}: {e}")

        if parts:
            combined = pd.concat(parts, ignore_index=True)
            combined.to_csv(RFF_COMBINED, index=False)
            log.info(f"Saved combined RFF CSV: {RFF_COMBINED} | {combined.shape}")
        else:
            log.error("No RFF files downloaded.")
    except Exception as e:
        log.error(f"RFF GitHub API failed: {e}")
else:
    log.info(f"RFF combined file exists: {RFF_COMBINED}")

In [ ]:
# ── Load and Clean RFF Carbon Pricing Data ─────────────────────────────────
# Columns in per-country files:
#   jurisdiction, year, ipcc_code, Product, tax (binary), ets (binary),
#   tax_rate_incl_ex_clcu (local currency/tCO2e), ets_price (local currency/tCO2e)

def load_rff_carbon(path: Path) -> pd.DataFrame:
    """
    Load combined RFF World Carbon Pricing Database CSV.
    Aggregates across sectors (ipcc_code) to get max carbon price per country-year.

    Output columns:
        iso3, year, tax_price_local, ets_price_local, has_carbon_price

    Note: prices are in domestic currency per tCO2e — not USD-converted.
    has_carbon_price = True if any mechanism active in that country-year.

    Args:
        path: Path to combined WorldCarbonPricingDatabase.csv

    Returns:
        Cleaned DataFrame. Empty DataFrame if file missing.

    Usage:
        carbon_df = load_rff_carbon(RFF_COMBINED)
    """
    if not path.exists():
        log.warning(f"RFF file not found at {path}.")
        return pd.DataFrame()

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        log.error(f"Could not parse RFF CSV: {e}")
        return pd.DataFrame()

    log.info(f"RFF raw: {df.shape} | columns: {list(df.columns)}")

    # Normalize country name → ISO-3
    country_col = next((c for c in df.columns if c in ("jurisdiction", "country")), None)
    if not country_col:
        log.error("No country/jurisdiction column found.")
        return pd.DataFrame()
    df = df.rename(columns={country_col: "country"})
    df = normalize_countries(df, "country")
    df = df.dropna(subset=["iso3"])

    # Year filter
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]

    # Tax rate: prefer tax_rate_incl_ex_clcu, fall back to tax binary flag
    if "tax_rate_incl_ex_clcu" in df.columns:
        df["tax_val"] = pd.to_numeric(df["tax_rate_incl_ex_clcu"], errors="coerce").fillna(0)
    elif "tax" in df.columns:
        df["tax_val"] = pd.to_numeric(df["tax"], errors="coerce").fillna(0)
    else:
        df["tax_val"] = 0

    # ETS price
    if "ets_price" in df.columns:
        df["ets_val"] = pd.to_numeric(df["ets_price"], errors="coerce").fillna(0)
    elif "ets" in df.columns:
        df["ets_val"] = pd.to_numeric(df["ets"], errors="coerce").fillna(0)
    else:
        df["ets_val"] = 0

    # Aggregate: max across sectors per country-year
    out = (
        df.groupby(["iso3", "year"])
        .agg(tax_price_local=("tax_val", "max"),
             ets_price_local=("ets_val", "max"))
        .reset_index()
    )
    out["has_carbon_price"] = (
        (out["tax_price_local"] > 0) | (out["ets_price_local"] > 0)
    )

    n_priced = out[out["has_carbon_price"]]["iso3"].nunique()
    log.info(f"RFF cleaned: {out.shape} | countries with carbon pricing: {n_priced}")
    return out[["iso3", "year", "tax_price_local", "ets_price_local", "has_carbon_price"]]


carbon_df = load_rff_carbon(RFF_COMBINED)

if not carbon_df.empty:
    display(carbon_df[carbon_df["has_carbon_price"]].sort_values(["iso3", "year"]).head(15))

In [7]:
# ── Save All ST3 Outputs to Parquet ────────────────────────────────────────
# All ST3 processed files go to data/processed/ST3/

if not st3_energy.empty:
    save_parquet(st3_energy, ST3_PROC / "st3_energy.parquet", "ST3 Energy Mix")
else:
    log.warning("st3_energy is empty — check OWID download.")
    stub = pd.DataFrame(columns=[
        "iso3", "year", "renewables_share", "fossil_share", "nuclear_share",
        "solar_twh", "wind_twh", "hydro_twh", "electricity_twh", "primary_energy_twh"
    ])
    save_parquet(stub, ST3_PROC / "st3_energy.parquet", "ST3 Energy Mix (stub)")

if not carbon_df.empty:
    save_parquet(carbon_df, ST3_PROC / "st3_carbon_price.parquet", "ST3 Carbon Pricing")
else:
    log.warning("carbon_df is empty — check RFF download.")
    stub = pd.DataFrame(columns=[
        "iso3", "year", "tax_price_local", "ets_price_local", "has_carbon_price"
    ])
    save_parquet(stub, ST3_PROC / "st3_carbon_price.parquet", "ST3 Carbon Pricing (stub)")

log.info("Phase 1 ST3 complete. Outputs in data/processed/ST3/")

17:02:42 [INFO] Saved ST3 Energy Mix: 2,793 rows × 10 cols → /Users/taruntheegela/Desktop/lambda/data/processed/ST3/st3_energy.parquet
17:02:42 [WARNING] carbon_df is empty — check RFF download.
17:02:42 [INFO] Saved ST3 Carbon Pricing (stub): 0 rows × 5 cols → /Users/taruntheegela/Desktop/lambda/data/processed/ST3/st3_carbon_price.parquet
17:02:42 [INFO] Phase 1 ST3 complete. Outputs in data/processed/ST3/
